# MEG Single-Word Decoding — Multi-Configuration Trainer

This notebook trains **five** CNN-encoder × LLM-decoder ensemble configurations back-to-back on a Colab A100 high-RAM runtime, runs per-configuration Bayesian HPO, and aggregates per-config figures and evaluation results into a single summary table on Google Drive.

**Configurations trained (in order):**
1. `simpleconv_t5large`       — SimpleConv CNN + T5-large brain transformer
2. `simpleconv_llama8b`       — SimpleConv CNN + LLaMA 3.1 8B brain transformer
3. `multiscaleconv_t5large`   — MultiScale SimpleConv CNN + T5-large brain transformer
4. `multiscaleconv_llama8b`   — MultiScale SimpleConv CNN + LLaMA 3.1 8B brain transformer
5. `multiscaleconv_pretrain_llama8b` — MultiScale SimpleConv with MAE-EEG pretraining + LLaMA 3.1 8B

**One-time setup (do this before running):**
1. Open the shared Drive folder: https://drive.google.com/drive/folders/1HOvzJEv0Czn-yJKELq6kp5cORJ5yBTdM
2. Right-click → **Organize** → **Add shortcut to Drive** → place it in **My Drive** named exactly `single-word-decoding`.

**Workflow**
1. Run initial setup (Section 1, steps 1.1 – 1.5).
2. Run training (Section 2, steps 2.1 – 2.3). HPO can be skipped by setting `RUN_HPO = False` in step 2.1.
3. Aggregate results (Section 3).

**Data / results strategy**
- Training happens entirely on the local A100 runtime disk (fast).
- Only figures and the aggregated evaluation-results table are persisted to Drive.
- Checkpoints, Lightning logs, caches, and job files are dropped between configs so the 100 GB instance disk never saturates.

## Section 1: Setup

---

Imports data, installs dependencies, builds the repository, and defines the set of configurations to train.

### Step 1.1: Mount Drive & Authenticate

Run this first. Colab will ask you to sign in and authorize access. HuggingFace authentication is required for the gated LLaMA 3.1 8B weights — add your token to Colab **Secrets** as `HF_TOKEN` beforehand.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login
import os, sys, shutil, torch
drive.mount('/content/drive')

# Auth HuggingFace for gated llama weights
token = userdata.get('HF_TOKEN')
login(token)

# ── Branch ─────────────────────────────────────────────────────────────
BRANCH         = 'multi-timescale-CNN'

# ── Paths ──────────────────────────────────────────────────────────────
TRAIN_SUBJECTS = ['sub-01','sub-02','sub-03','sub-04','sub-05','sub-06','sub-07','sub-08','sub-09','sub-10',
                  'sub-11','sub-12','sub-13','sub-14','sub-15','sub-16','sub-17','sub-18','sub-19','sub-20',
                  'sub-21','sub-22','sub-23','sub-24','sub-25','sub-26','sub-27']

REPO           = 'https://github.com/kazumah1/single-word-decoding.git'
WORKDIR        = f'/content/single-word-decoding-{BRANCH.replace("/", "-")}'
LOCAL_DATAPATH = '/content/neural_data'

# Where the Gwilliams 2022 MEG data already lives on Drive
DRIVE_GW       = '/content/drive/MyDrive/datasets/gwilliams2022/download'

# Drive path for *final* outputs only (figures + aggregated eval CSV).
DRIVE_SAVEPATH = f'/content/drive/MyDrive/sentence-results/{BRANCH.replace("/", "-")}/multi_config'
# Local temporary path — all training intermediates (checkpoints, logs, caches) stay here.
SAVEPATH       = f'/content/tmp_training_output/{BRANCH.replace("/", "-")}'
# Per-config artifact aggregator inside the local save path.
MULTI_CFG_OUT  = f'{SAVEPATH}/multi_config_runs'
# ───────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_SAVEPATH, exist_ok=True)
os.makedirs(f'{DRIVE_SAVEPATH}/hpo', exist_ok=True)
os.makedirs(f'{DRIVE_SAVEPATH}/multi_config_runs', exist_ok=True)
os.makedirs(f'{DRIVE_SAVEPATH}/summary', exist_ok=True)
os.makedirs(SAVEPATH, exist_ok=True)
os.makedirs(f'{SAVEPATH}/cache', exist_ok=True)
os.makedirs(MULTI_CFG_OUT, exist_ok=True)
os.makedirs(LOCAL_DATAPATH, exist_ok=True)

print(f'Branch: {BRANCH}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
total, _, free = shutil.disk_usage('/content')
print(f'Disk:   {free/1e9:.0f} GB free / {total/1e9:.0f} GB total')
print(f'\nTrain subjects: {TRAIN_SUBJECTS}')
print(f'Local SAVEPATH:      {SAVEPATH}')
print(f'Local MULTI_CFG_OUT: {MULTI_CFG_OUT}')
print(f'Final Drive output:  {DRIVE_SAVEPATH}')

### Step 1.2: Instance Setup (run once per session)

Clones the `multi-timescale-CNN` branch, installs all dependencies, and transfers the MEG data from Drive into the local runtime:

In [ ]:
import os, shutil, subprocess, sys

# ── 1. Clone repo ──────────────────────────────────────────────────────
print(f'=== Cloning branch: {BRANCH} ===')
if os.path.exists(WORKDIR):
    print('  Already cloned, pulling latest...')
    subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
if not os.path.exists(f'{WORKDIR}/sentence_results'):
    os.symlink(SAVEPATH, f'{WORKDIR}/sentence_results')
os.makedirs(f'{WORKDIR}/projects', exist_ok=True)
print('  Done.')

# ── 2. Install local packages ──────────────────────────────────────────
print('\n=== Installing local packages ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuralset/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuraltrain/'], check=True)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print('  Done.')

# ── 3. Install dependencies ────────────────────────────────────────────
print('\n=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightning', 'pytorch-lightning', 'torchvision',
                'wandb', 'osfclient', 'mne_bids', 'tqdm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'x-transformers==1.26.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics==1.5.2'], check=True)
print('  Done.')

# ── 4. kenlm ──────────────────────────────────────────────────────────
print('\n=== Installing kenlm ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kenlm'], capture_output=True)
try:
    import kenlm; print('  Installed from PyPI.')
except ImportError:
    print('  PyPI failed — building from source...')
    subprocess.run(['git', 'clone', 'https://github.com/kpu/kenlm', '/content/kenlm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cython'], check=True)
    subprocess.run(['cython', '/content/kenlm/python/kenlm.pyx', '--cplus'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '/content/kenlm'], check=True)
    import kenlm; print('  Built from source.')

# ── 5. Copy training subjects from Drive to local runtime ──────────────
print('\n=== Copying data from Drive to local disk ===')
local_gw = f'{LOCAL_DATAPATH}/gwilliams2022/download'
os.makedirs(local_gw, exist_ok=True)

for item in ['stimuli'] + TRAIN_SUBJECTS:
    src = f'{DRIVE_GW}/{item}'
    dst = f'{local_gw}/{item}'
    if os.path.exists(dst):
        print(f'  {item} already on local disk, skipping.')
    elif os.path.exists(src):
        print(f'  Copying {item}...')
        shutil.copytree(src, dst)
    else:
        print(f'  WARNING: {item} not found at {src}')

_, _, free = shutil.disk_usage('/content')
print(f'  Disk free: {free/1e9:.0f} GB')
print(f'\n✓ Setup complete.')

### Step 1.3: Patch Repository Sources

Two small source patches that this multi-config pipeline depends on:

- **`multiscaleconv.py`** — removes duplicate `SimpleConv*` class definitions and re-exports them via `from .simpleconv import …` so both backbone variants share a single source of truth.
- **`sentence_decoding/main.py`** — fixes a pre-existing bug in the `ModelCheckpoint` block of `Experiment.fit()` where `monitor=monitor, mode=monitor_mode` reference undefined local variables. This would crash any run with `save_checkpoints=True`, which the pretrain recipe in Step 2.3 requires.

In [ ]:
import re

# ---------------------------------------------------------------------- #
# A. Patch neuraltrain/.../multiscaleconv.py so the SimpleConv* classes   #
#    (ConvSequence, SpatialFilter, SimpleConvConfig, SimpleConv,         #
#    SimpleConvTimeAggConfig, SimpleConvTimeAgg) can be imported from it.#
# ---------------------------------------------------------------------- #
path = f'{WORKDIR}/neuraltrain/neuraltrain/models/multiscaleconv.py'
with open(path) as f:
    src = f.read()

# 1. Add imports from simpleconv after the transformer import line (if not already present)
if 'from .simpleconv import (' not in src:
    src = src.replace(
        'from .transformer import LlamaTransformerConfig, TransformerEncoderConfig',
        'from .transformer import LlamaTransformerConfig, TransformerEncoderConfig\n'
        'from .simpleconv import (\n'
        '    ConvSequence, SpatialFilter,\n'
        '    SimpleConvConfig, SimpleConv,\n'
        '    SimpleConvTimeAggConfig, SimpleConvTimeAgg,\n'
        ')'
    )

# 2. Remove the duplicate class blocks for SimpleConvConfig, SimpleConv,
#    SimpleConvTimeAggConfig, SimpleConvTimeAgg — everything between
#    MultiScaleConvSequence and MultiScaleSimpleConvConfig
src = re.sub(
    r'(\n# -{5,}\n# MultiScaleConvSequence.*?)\n# -{5,}\n# SimpleConvConfig.*?\n# -{5,}\n# MultiScaleSimpleConvConfig',
    r'\1\n\n# ---------------------------------------------------------------------------\n# MultiScaleSimpleConvConfig',
    src, flags=re.DOTALL
)

# 3. Remove duplicate SimpleConvTimeAggConfig and SimpleConvTimeAgg blocks
#    (between MultiScaleSimpleConv and MultiScaleSimpleConvTimeAgg)
src = re.sub(
    r'(\nclass MultiScaleSimpleConv\b.*?)\nclass SimpleConvTimeAggConfig\b.*?\nclass MultiScaleSimpleConvTimeAggConfig',
    r'\1\n\nclass MultiScaleSimpleConvTimeAggConfig',
    src, flags=re.DOTALL
)

with open(path, 'w') as f:
    f.write(src)

# Verify
classes = [l for l in src.splitlines() if l.startswith('class ')]
print('multiscaleconv.py classes:')
for c in classes:
    print(' ', c)

# ---------------------------------------------------------------------- #
# B. Patch sentence_decoding/main.py — the ModelCheckpoint block in fit() #
#    references undefined locals `monitor` / `monitor_mode`, which crashes#
#    any run with save_checkpoints=True (needed for the pretrain recipe).#
# ---------------------------------------------------------------------- #
main_path = f'{WORKDIR}/sentence_decoding/main.py'
with open(main_path) as f:
    msrc = f.read()

if 'monitor=monitor,' in msrc and 'mode=monitor_mode,' in msrc:
    msrc = msrc.replace(
        'monitor=monitor,\n                    mode=monitor_mode,',
        'monitor=self.trainer_config.monitor,\n'
        '                    mode="max" if "acc" in self.trainer_config.monitor else "min",',
    )
    with open(main_path, 'w') as f:
        f.write(msrc)
    print('main.py: patched ModelCheckpoint(monitor=..., mode=...).')
else:
    print('main.py: ModelCheckpoint already patched (or different source layout) — skipped.')

### Step 1.4: Define Configurations to Train

Declares the ordered list of configurations this notebook will train. `CONFIGS_TO_TRAIN` is the single source of truth; Section 2 HPO and Section 2.3 training both iterate this list. To skip a configuration, comment out its entry.

The per-config deltas are defined in `sentence_decoding/grids/multi_config.py` (helper module shipped with the branch). Each delta sets the CNN backbone (`brain_model_config.name`), the feature-extractor model (`data.feature.model_name`), and — for the T5 variants — swaps the brain transformer to `TransformerEncoder` (since `LlamaTransformer` is LLaMA-specific).

In [ ]:
# Ensure the cloned repo is importable
import sys
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

# Pull the canonical per-config deltas from the repo
from sentence_decoding.grids.multi_config import (
    CONFIG_DELTAS,
    free_memory,
    cleanup_run_artifacts as _cleanup_run_dir,
    aggregate_eval_results,
    plot_combined_loss_curves,
)

# Order matters: earliest first. Disable a run by commenting its line.
CONFIGS_TO_TRAIN = [
    'simpleconv_t5large',
    'simpleconv_llama8b',
    'multiscaleconv_t5large',
    'multiscaleconv_llama8b',
    'multiscaleconv_pretrain_llama8b',
]

# Sanity-check that every requested config has a delta entry
missing = [c for c in CONFIGS_TO_TRAIN if c not in CONFIG_DELTAS]
assert not missing, f'Unknown config(s): {missing}. Known: {list(CONFIG_DELTAS.keys())}'

print(f'Will train {len(CONFIGS_TO_TRAIN)} configuration(s):')
for i, name in enumerate(CONFIGS_TO_TRAIN, 1):
    delta = CONFIG_DELTAS[name]
    brain = delta.get('brain_model_config.name', '(default)')
    feat  = delta.get('data.feature.model_name', '(default)')
    print(f'  {i}. {name:<38s}  brain={brain:<32s}  feature={feat}')

### Step 1.5: Clean Local Artifacts (Optional)

Drops any previous per-config runs and the aggregated output directory so a fresh multi-config sweep starts clean. Skip if you want to resume.

The shared dataset cache (`SAVEPATH/cache`) is **preserved** — deleting it would re-trigger the ~30 min `StudyLoader` build on the next run.

In [ ]:
import shutil, os

# Drop all per-run directories (checkpoints, logs) and the aggregated output
for sub in ('runs', 'multi_config_runs', 'hpo', 'results'):
    p = os.path.join(SAVEPATH, sub)
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f'Removed {p}')

os.makedirs(MULTI_CFG_OUT, exist_ok=True)
os.makedirs(f'{SAVEPATH}/hpo', exist_ok=True)
os.makedirs(f'{SAVEPATH}/runs', exist_ok=True)
print('\nLocal artifact dirs reset. Dataset cache preserved.')

## Section 2: Training

---

Runs per-configuration Bayesian hyperparameter search (Section 2.2), then trains every configuration in `CONFIGS_TO_TRAIN` end-to-end with the best-found HPs (Section 2.3). HPO figures go to `DRIVE_SAVEPATH/hpo/<config_name>/`; per-config training artifacts (figures + `eval_results.json` + `training_history.csv`) go to `MULTI_CFG_OUT/<config_name>/` and are mirrored to Drive incrementally.

### Step 2.1: HPO & Training Setup

Installs Optuna, defines the shared Bayesian search space, and specifies the per-trial budget (`HPO_CONFIG`). Set `RUN_HPO = False` to skip HPO and fall back to the defaults baked into `defaults.py` + `CONFIG_DELTAS`.

In [ ]:
# ============================================================
# Multi-config HPO — Cell A: install deps & define search space
# ============================================================
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'optuna>=3.6', 'plotly>=5.20', 'kaleido==0.2.1'], check=True)

# Env vars must be set BEFORE importing sentence_decoding (defaults.py reads them at import time)
os.environ['DATAPATH']   = LOCAL_DATAPATH
os.environ['SAVEPATH']   = SAVEPATH
os.environ['WANDB_MODE'] = 'disabled'
os.environ['PYTHONPATH'] = WORKDIR
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
for pkg in ['neuralset', 'neuraltrain']:
    pkg_path = os.path.join(WORKDIR, pkg)
    if pkg_path not in sys.path:
        sys.path.insert(0, pkg_path)

# ---- Toggle HPO on / off ---------------------------------------------
RUN_HPO = True   # set False to skip and use defaults for every config

# ---- Search space ----------------------------------------------------
# Same set of CNN-encoder + optimizer HPs applies to SimpleConv and
# MultiScaleSimpleConv (they share backbone parameterization).
SEARCH_SPACE = {
    'lr':                {'type': 'loguniform', 'low': 1e-5, 'high': 5e-3,
                          'cfg': 'trainer_config.lr'},
    'weight_decay':      {'type': 'loguniform', 'low': 1e-7, 'high': 1e-2,
                          'cfg': 'trainer_config.weight_decay'},
    'gradient_clip_val': {'type': 'uniform',    'low': 0.0,  'high': 5.0,
                          'cfg': 'trainer_config.gradient_clip_val'},
    'dropout_input':     {'type': 'uniform',    'low': 0.0,  'high': 0.5,
                          'cfg': 'brain_model_config.dropout_input'},
    'hidden':            {'type': 'categorical','choices': [128, 160, 256, 320],
                          'cfg': 'brain_model_config.hidden'},
    'depth':             {'type': 'int',        'low': 3, 'high': 7,
                          'cfg': 'brain_model_config.depth'},
    'kernel_size':       {'type': 'categorical','choices': [3, 5, 7],
                          'cfg': 'brain_model_config.kernel_size'},
    'dilation_period':   {'type': 'int',        'low': 3, 'high': 7,
                          'cfg': 'brain_model_config.dilation_period'},
    'glu':               {'type': 'int',        'low': 1, 'high': 3,
                          'cfg': 'brain_model_config.glu'},
    'initial_linear':    {'type': 'categorical','choices': [256, 512, 768],
                          'cfg': 'brain_model_config.initial_linear'},
    'spatial_filters':   {'type': 'categorical','choices': [16, 32, 64],
                          'cfg': 'brain_model_config.spatial_filters'},
    'batch_size':        {'type': 'categorical','choices': [64, 128, 256],
                          'cfg': 'data.batch_size'},
}

# ---- Per-config HPO budget -------------------------------------------
# Each trial is aggressively down-scaled so the 8B-param ensemble stays
# tractable on a single A100: use_transformer=False skips the frozen
# decoder (dominant cost), and the CNN encoder is the under-trained
# component anyway. Trial count is per-config; with 5 configs, 8 trials
# each ≈ 40 trials total.
HPO_CONFIG = {
    'n_trials':              8,       # trials PER config (raise for more coverage)
    'max_epochs_per_trial':  5,
    'n_timelines_per_trial': 800,
    'n_subjects_per_trial':  6,
    'use_transformer_in_search': False,
    'proxy_metric':          'val_contrastive_top5_acc',
    'proxy_mode':            'max',
    'pruner_warmup_epochs':  2,
    'pruner_min_trials':     3,
    'storage_dir':           f'{SAVEPATH}/hpo',
    'seed':                  0,
    # Configs that reuse another config's HPs (pretrain shares architecture
    # with non-pretrain counterpart, so no need to re-tune).
    'hpo_alias': {
        'multiscaleconv_pretrain_llama8b': 'multiscaleconv_llama8b',
    },
}
os.makedirs(HPO_CONFIG['storage_dir'], exist_ok=True)

print(f'Tunable HPs ({len(SEARCH_SPACE)}): {list(SEARCH_SPACE.keys())}')
print(f"Trials/config: {HPO_CONFIG['n_trials']}   Epochs/trial: {HPO_CONFIG['max_epochs_per_trial']}")
print(f"Proxy metric:  {HPO_CONFIG['proxy_metric']} ({HPO_CONFIG['proxy_mode']})")
print(f"HPO results:   {HPO_CONFIG['storage_dir']}")
print(f"RUN_HPO:       {RUN_HPO}")

### Step 2.2: Per-Configuration Bayesian Hyperparameter Search

For every entry in `CONFIGS_TO_TRAIN`, runs an independent TPE study with median pruning. HPO is performed with the brain transformer disabled (dominant cost) so the search targets the CNN encoder + optimizer. Best parameters, the full trials table, and interactive Plotly figures are written to `DRIVE_SAVEPATH/hpo/<config_name>/`.

Configs listed in `HPO_CONFIG['hpo_alias']` reuse the best HPs from their alias target instead of re-tuning (architecture-equivalent variants).

In [ ]:
# ============================================================
# Multi-config HPO — Cell B: per-config TPE + MedianPruner
# ============================================================
import os, gc, time, json, copy, warnings, shutil
import torch
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback, LearningRateMonitor, EarlyStopping

from sentence_decoding.main import Experiment
from sentence_decoding.grids.defaults import default_config
from neuraltrain.utils import update_config

warnings.filterwarnings('ignore', category=UserWarning)

MONITOR = HPO_CONFIG['proxy_metric']
MODE    = HPO_CONFIG['proxy_mode']

BEST_HPS: dict[str, dict] = {}        # populated by this cell, read by 2.3
STUDIES:  dict[str, 'optuna.Study'] = {}  # kept for 2.3 figure re-render

# ---- Lightning callback that pipes val metric to Optuna each epoch ----
class OptunaPruneCallback(Callback):
    def __init__(self, trial, monitor):
        self.trial, self.monitor = trial, monitor
    def on_validation_epoch_end(self, trainer, pl_module):
        cm = trainer.callback_metrics
        val = cm.get(self.monitor, None)
        if val is None:  # Lightning may append a dataloader suffix
            hits = [v for k, v in cm.items() if k.startswith(self.monitor)]
            val = hits[0] if hits else None
        if val is None:
            return
        self.trial.report(float(val), step=trainer.current_epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()

def _suggest(trial, name, spec):
    t = spec['type']
    if t == 'loguniform':  return trial.suggest_float(name, spec['low'], spec['high'], log=True)
    if t == 'uniform':     return trial.suggest_float(name, spec['low'], spec['high'])
    if t == 'int':         return trial.suggest_int(name, spec['low'], spec['high'])
    if t == 'categorical': return trial.suggest_categorical(name, spec['choices'])
    raise ValueError(t)

def _read_monitor(trainer):
    cm = trainer.callback_metrics
    if MONITOR in cm: return float(cm[MONITOR])
    hits = [v for k, v in cm.items() if k.startswith(MONITOR)]
    return float(hits[0]) if hits else (float('-inf') if MODE == 'max' else float('inf'))

# ---- HPO objective factory -------------------------------------------
def _make_objective(config_name: str, study_dir: str):
    base_delta = CONFIG_DELTAS[config_name]
    def objective(trial):
        torch.cuda.empty_cache(); gc.collect()
        sample = {n: _suggest(trial, n, s) for n, s in SEARCH_SPACE.items()}
        overrides = {
            'seed': HPO_CONFIG['seed'],
            'config_name': f'{config_name}_hpo_trial{trial.number:03d}',
            'infra.cluster': None,
            'use_wandb': False,
            'save_checkpoints': False,
            'use_transformer': HPO_CONFIG['use_transformer_in_search'],
            'data.n_timelines': HPO_CONFIG['n_timelines_per_trial'],
            'data.n_subjects':  HPO_CONFIG['n_subjects_per_trial'],
            'data.num_workers': 4,
            'trainer_config.n_epochs': HPO_CONFIG['max_epochs_per_trial'],
            'trainer_config.patience': max(2, HPO_CONFIG['max_epochs_per_trial'] // 2),
            'trainer_config.monitor':  MONITOR,
            'trainer_config.transformer_start_epoch': 10_000,
        }
        for name, val in sample.items():
            overrides[SEARCH_SPACE[name]['cfg']] = val
        # Apply architecture delta FIRST, then HPO overrides on top
        cfg = update_config(default_config, base_delta)
        cfg = update_config(cfg, overrides)
        cfg['infra']['folder'] = os.path.join(study_dir, 'trials', f'trial_{trial.number:03d}')
        os.makedirs(cfg['infra']['folder'], exist_ok=True)

        t0 = time.time()
        task = Experiment(**cfg)
        task.infra.clear_job()
        loaders = task.setup_run()
        brain_model, transformer = task.get_model(loaders['train'])
        module = task.load_module(brain_model, transformer)

        callbacks = [
            LearningRateMonitor(logging_interval='epoch'),
            EarlyStopping(monitor=MONITOR,
                          patience=task.trainer_config.patience,
                          mode=MODE, verbose=False, check_finite=False),
            OptunaPruneCallback(trial, MONITOR),
        ]
        trainer = pl.Trainer(
            gradient_clip_val=task.trainer_config.gradient_clip_val,
            devices=task.infra.gpus_per_node,
            max_epochs=task.trainer_config.n_epochs,
            enable_progress_bar=False,
            log_every_n_steps=50,
            logger=False,
            callbacks=callbacks,
            inference_mode=False,
            enable_checkpointing=False,
            num_sanity_val_steps=0,
        )
        pl.seed_everything(task.seed, verbose=False)
        try:
            trainer.fit(module,
                        train_dataloaders=loaders['train'],
                        val_dataloaders=loaders['val'])
            score = _read_monitor(trainer)
        finally:
            del task, brain_model, transformer, module, trainer, loaders
            # Trial dir is only useful transiently; drop to keep disk free.
            shutil.rmtree(cfg['infra']['folder'], ignore_errors=True)
            torch.cuda.empty_cache(); gc.collect()

        print(f'  [trial {trial.number:03d}] {MONITOR}={score:.4f}  '
              f'({time.time()-t0:.0f}s)  sample={sample}')
        return score
    return objective

# ---- HPO figure saver ------------------------------------------------
def _save_hpo_figures(study, out_dir: str, proxy_metric: str, proxy_mode: str):
    import optuna.visualization as vis
    import pandas as pd
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    fig_dir = os.path.join(out_dir, 'figures')
    os.makedirs(fig_dir, exist_ok=True)

    # Interactive Plotly figures (written as PNG via kaleido)
    for title, fn in [
        ('optimization_history',  vis.plot_optimization_history),
        ('parameter_importances', vis.plot_param_importances),
        ('parallel_coordinate',   vis.plot_parallel_coordinate),
        ('slice',                 vis.plot_slice),
        ('intermediate_values',   vis.plot_intermediate_values),
        ('edf',                   vis.plot_edf),
    ]:
        try:
            fig = fn(study)
            fig.update_layout(title=title.replace('_', ' ').title())
            fig.write_image(os.path.join(fig_dir, f'{title}.png'),
                            width=1000, height=600, scale=2)
        except Exception as e:
            print(f'  skipped {title}: {e}')

    # Summary matplotlib figure
    completed = [t for t in study.trials if t.value is not None]
    pruned    = [t for t in study.trials if t.state.name == 'PRUNED']
    if completed:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
        vals = [t.value for t in completed]
        running_best = []
        best = -float('inf') if proxy_mode == 'max' else float('inf')
        for v in vals:
            best = max(best, v) if proxy_mode == 'max' else min(best, v)
            running_best.append(best)
        axes[0].plot(range(len(vals)), vals, 'o', alpha=0.5, label='trial score')
        axes[0].plot(range(len(running_best)), running_best, '-', lw=2, label='running best')
        axes[0].set_xlabel('completed trial'); axes[0].set_ylabel(proxy_metric)
        axes[0].set_title(f'Optimization progress  ({len(pruned)} pruned)')
        axes[0].legend(); axes[0].grid(alpha=0.3)
        try:
            imp = optuna.importance.get_param_importances(study)
            top_params = list(imp.keys())[:4]
        except Exception:
            top_params = list(study.best_params.keys())[:4]
        ax = axes[1]
        for p in top_params:
            ys = [t.value for t in completed if p in t.params]
            ax.scatter(range(len(ys)), ys, label=p, alpha=0.7)
        ax.set_xlabel('trial order'); ax.set_ylabel(proxy_metric)
        ax.set_title('Top-importance HPs vs score'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'summary.png'), dpi=150, bbox_inches='tight')
        plt.close(fig)

    # Trials table
    df = study.trials_dataframe(attrs=('number','value','state','duration','params'))
    df = df.sort_values('value', ascending=(proxy_mode != 'max'))
    df.to_csv(os.path.join(out_dir, 'trials.csv'), index=False)

    # Best params
    with open(os.path.join(out_dir, 'best_params.json'), 'w') as f:
        json.dump({
            'metric':      proxy_metric,
            'mode':        proxy_mode,
            'best_value':  study.best_value,
            'best_trial':  study.best_trial.number,
            'best_params': study.best_params,
            'n_trials':    len(study.trials),
            'n_pruned':    len(pruned),
        }, f, indent=2)

# ---- Main per-config HPO loop ----------------------------------------
alias_map = HPO_CONFIG.get('hpo_alias', {})

if not RUN_HPO:
    print('RUN_HPO=False — skipping HPO, will use baseline HPs for every config.')
else:
    for cfg_name in CONFIGS_TO_TRAIN:
        # Reuse another config's HPs if aliased
        if cfg_name in alias_map:
            src = alias_map[cfg_name]
            if src in BEST_HPS:
                BEST_HPS[cfg_name] = dict(BEST_HPS[src])
                print(f'[{cfg_name}] aliases {src} — reusing its best HPs, no search run.')
                continue
            elif src not in CONFIGS_TO_TRAIN:
                print(f'[{cfg_name}] aliases {src} but it is not in CONFIGS_TO_TRAIN; running own HPO.')

        study_dir = os.path.join(HPO_CONFIG['storage_dir'], cfg_name)
        os.makedirs(study_dir, exist_ok=True)
        storage = f'sqlite:///{study_dir}/study.db'
        sampler = TPESampler(seed=HPO_CONFIG['seed'], multivariate=True,
                             group=True, constant_liar=True, n_startup_trials=3)
        pruner  = MedianPruner(n_startup_trials=HPO_CONFIG['pruner_min_trials'],
                               n_warmup_steps=HPO_CONFIG['pruner_warmup_epochs'])
        study = optuna.create_study(
            study_name=f'hpo_{cfg_name}',
            direction='maximize' if MODE == 'max' else 'minimize',
            storage=storage, load_if_exists=True,
            sampler=sampler, pruner=pruner,
        )
        print(f'\n{"="*70}\n[{cfg_name}]  running {HPO_CONFIG["n_trials"]} trials\n{"="*70}')
        t0 = time.time()
        study.optimize(
            _make_objective(cfg_name, study_dir),
            n_trials=HPO_CONFIG['n_trials'],
            gc_after_trial=True,
            show_progress_bar=False,
            catch=(RuntimeError,),
        )
        elapsed = (time.time() - t0) / 60.0
        print(f'[{cfg_name}] done in {elapsed:.1f} min  '
              f'best_trial=#{study.best_trial.number}  {MONITOR}={study.best_value:.4f}')
        for k, v in study.best_params.items():
            print(f'    {k:<20s} = {v}')

        BEST_HPS[cfg_name] = dict(study.best_params)
        STUDIES[cfg_name] = study

        # Save HPO figures + summary straight to Drive
        _save_hpo_figures(study, study_dir, MONITOR, MODE)
        drive_hpo_dir = os.path.join(DRIVE_SAVEPATH, 'hpo', cfg_name)
        if os.path.exists(drive_hpo_dir):
            shutil.rmtree(drive_hpo_dir, ignore_errors=True)
        shutil.copytree(study_dir, drive_hpo_dir,
                        ignore=shutil.ignore_patterns('*.db', 'trials'))
        print(f'[{cfg_name}] HPO figures → {drive_hpo_dir}')

        # Free memory between studies
        del study, sampler, pruner
        free_memory()

print('\n' + '='*70)
print('HPO complete — best HPs per config:')
print('='*70)
for name, hp in BEST_HPS.items():
    print(f'{name}: {hp}')

### Step 2.3: Train All Configurations

Trains every entry in `CONFIGS_TO_TRAIN` in sequence with the best HPs found in Step 2.2 (or the defaults if HPO was skipped). After each config:

1. `task.collect_run_artifacts(MULTI_CFG_OUT)` copies the figures, `eval_results.json`, and `training_history.csv` into `MULTI_CFG_OUT/<config_name>/`.
2. `task.cleanup_run_artifacts()` drops checkpoints, caches, Lightning logs, and job files so the A100 disk does not saturate.
3. `free_memory()` empties the CUDA cache and triggers a full GC pass.
4. The per-config artifacts are mirrored to Drive immediately, so a session disconnect does not lose prior configs.

`multiscaleconv_pretrain_llama8b` is handled as a two-stage recipe: stage 1 runs SimCLR contrastive pretraining on `MultiScaleSimpleConvPretrain` for `PRETRAIN_EPOCHS` epochs to produce a checkpoint; stage 2 loads that checkpoint into `MultiScaleSimpleConvTimeAgg` and runs the normal supervised fit. The stage-1 checkpoint is deleted after stage 2 completes. (MAE-EEG is not used here because `Experiment.run()` would still call `.test()` for it, which requires a supervised retrieval head.)

In [ ]:
# ============================================================
# Multi-config training loop
# ============================================================
import os, shutil, json, time, copy, traceback, gc
import torch
from sentence_decoding.main import Experiment
from sentence_decoding.grids.defaults import default_config
from neuraltrain.utils import update_config

# ---- Knobs you might want to tweak ----------------------------------
# Epochs for the full supervised fit. defaults.py ships with 50; the
# monitor/patience values in trainer_config then control early stopping.
TRAIN_EPOCHS       = 50
PRETRAIN_EPOCHS    = 20          # stage-1 epochs for the pretrain recipe
# Only 'simclr' is fully plumbed on this branch: it has its own dataset
# (ContrastiveSegmentDataset), its own module (SimCLRModule), and is the
# only mode that Experiment.run() skips .test() for. 'maeeg' would fall
# through to retrieval .test() which requires a supervised head.
PRETRAIN_MODE      = 'simclr'

# Keep dataset-level knobs fixed across configs (defaults.py already sets these)
COMMON_OVERRIDES = {
    'infra.cluster': None,
    'use_wandb': False,
    'save_checkpoints': False,      # NEVER persist .ckpt from the main run
    'trainer_config.n_epochs': TRAIN_EPOCHS,
    'trainer_config.fast_dev_run': False,
    'data.dataset': 'Gwilliams2022',
}

# Ensure output dirs exist
os.makedirs(MULTI_CFG_OUT, exist_ok=True)
os.makedirs(f'{SAVEPATH}/runs', exist_ok=True)

def _apply_best_hps(overrides: dict, cfg_name: str) -> dict:
    hps = BEST_HPS.get(cfg_name) or {}
    for hp_name, hp_val in hps.items():
        overrides[SEARCH_SPACE[hp_name]['cfg']] = hp_val
    return overrides

def _run_single_config(cfg_name: str):
    """Train one configuration end-to-end. Collects artifacts + cleans up."""
    run_dir = os.path.join(SAVEPATH, 'runs', cfg_name)
    if os.path.exists(run_dir):
        shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir, exist_ok=True)

    delta = CONFIG_DELTAS[cfg_name]
    overrides = copy.deepcopy(COMMON_OVERRIDES)
    overrides['config_name'] = cfg_name
    overrides = _apply_best_hps(overrides, cfg_name)

    # ---------- special handling: two-stage pretrain recipe ---------- #
    pretrain_ckpt = None
    if cfg_name == 'multiscaleconv_pretrain_llama8b':
        pre_run_dir = os.path.join(SAVEPATH, 'runs', f'{cfg_name}__pretrain')
        if os.path.exists(pre_run_dir):
            shutil.rmtree(pre_run_dir, ignore_errors=True)
        os.makedirs(pre_run_dir, exist_ok=True)

        pre_overrides = copy.deepcopy(COMMON_OVERRIDES)
        pre_overrides = _apply_best_hps(pre_overrides, cfg_name)
        pre_overrides.update({
            'config_name': f'{cfg_name}__pretrain',
            'pretrain_mode': PRETRAIN_MODE,
            'brain_model_config.name': 'MultiScaleSimpleConvPretrain',
            'use_transformer': False,
            'save_checkpoints': True,        # stage-1 *must* save a ckpt
            'trainer_config.n_epochs': PRETRAIN_EPOCHS,
            # Pretraining has no retrieval metric; monitor the pretrain loss instead
            'trainer_config.monitor': 'val_loss',
        })
        pre_cfg = update_config(default_config, pre_overrides)
        pre_cfg['infra']['folder'] = pre_run_dir

        print(f'\n  [stage 1/2] pretrain ({PRETRAIN_MODE}, {PRETRAIN_EPOCHS} epochs) → {pre_run_dir}')
        t_pre = time.time()
        pre_task = Experiment(**pre_cfg)
        pre_task.infra.clear_job()
        pre_task.run()
        print(f'  [stage 1/2] done in {(time.time()-t_pre)/60:.1f} min')

        # Resolve the pretrain checkpoint path (prefer best.ckpt, fall back to last.ckpt)
        cand = [os.path.join(pre_run_dir, 'best.ckpt'),
                os.path.join(pre_run_dir, 'last.ckpt')]
        pretrain_ckpt = next((c for c in cand if os.path.exists(c)), None)
        if pretrain_ckpt is None:
            raise FileNotFoundError(f'No pretrain checkpoint found in {pre_run_dir}')
        # Move the ckpt out so we can delete the stage-1 folder entirely
        ckpt_stash = os.path.join(SAVEPATH, 'runs', f'{cfg_name}__pretrain.ckpt')
        shutil.move(pretrain_ckpt, ckpt_stash)
        pretrain_ckpt = ckpt_stash

        del pre_task
        shutil.rmtree(pre_run_dir, ignore_errors=True)
        free_memory()
        overrides['pretrain_checkpoint'] = pretrain_ckpt
        # stage 2 runs with pretrain_mode='none' (the default) and the finetune
        # delta (brain_model_config.name=MultiScaleSimpleConvTimeAgg, llama
        # transformer, etc.)

    # ---------- main supervised fit ---------------------------------- #
    cfg = update_config(default_config, delta)
    cfg = update_config(cfg, overrides)
    cfg['infra']['folder'] = run_dir

    print(f'  [train] {cfg_name}  → {run_dir}')
    print(f'    brain={cfg["brain_model_config"]["name"]}  '
          f'feature={cfg["data"]["feature"]["model_name"]}  '
          f'transformer={cfg["transformer_config"]["name"]}  '
          f'lr={cfg["trainer_config"]["lr"]}  bs={cfg["data"]["batch_size"]}')

    t0 = time.time()
    task = Experiment(**cfg)
    task.infra.clear_job()
    task.run()
    print(f'  [train] done in {(time.time()-t0)/60:.1f} min')

    # ---------- collect + cleanup ------------------------------------ #
    out_dir = task.collect_run_artifacts(MULTI_CFG_OUT, label=cfg_name)
    print(f'  [train] artifacts → {out_dir}')
    task.cleanup_run_artifacts()   # drop ckpts / lightning_logs etc.
    # Also drop the run directory entirely now that artifacts are copied
    shutil.rmtree(run_dir, ignore_errors=True)
    # Drop stashed pretrain checkpoint if any
    if pretrain_ckpt and os.path.exists(pretrain_ckpt):
        os.remove(pretrain_ckpt)

    # Mirror this config's artifacts to Drive immediately for safety
    drive_cfg_dir = os.path.join(DRIVE_SAVEPATH, 'multi_config_runs', cfg_name)
    if os.path.exists(drive_cfg_dir):
        shutil.rmtree(drive_cfg_dir, ignore_errors=True)
    if os.path.exists(out_dir):
        shutil.copytree(out_dir, drive_cfg_dir)
        print(f'  [train] mirrored to Drive → {drive_cfg_dir}')

    del task
    free_memory()

# ---------------------------------------------------------------------- #
print('='*70)
print(f'Training {len(CONFIGS_TO_TRAIN)} configuration(s)')
print('='*70)

overall_t0 = time.time()
failed = []
for i, cfg_name in enumerate(CONFIGS_TO_TRAIN, 1):
    print(f'\n[{i}/{len(CONFIGS_TO_TRAIN)}] {cfg_name}')
    print('-'*70)
    try:
        _run_single_config(cfg_name)
    except Exception as e:
        print(f'  !! {cfg_name} FAILED: {type(e).__name__}: {e}')
        traceback.print_exc()
        failed.append((cfg_name, str(e)))
        free_memory()

total_min = (time.time() - overall_t0) / 60.0
print('\n' + '='*70)
print(f'All configs finished in {total_min:.1f} min')
if failed:
    print(f'{len(failed)} failed:')
    for name, err in failed:
        print(f'  - {name}: {err}')
else:
    print('  All configs succeeded.')
print('='*70)

## Section 3: Results

---

Aggregates per-config `eval_results.json` into a single DataFrame, renders a combined loss-curves plot spanning all configurations, and writes the final deliverables (figures + `aggregated_eval_results.csv`) to `DRIVE_SAVEPATH/summary/`. No checkpoints, Lightning logs, HPO SQLite databases, or job files are persisted to Drive.

In [ ]:
# ============================================================
# Aggregate results + mirror final deliverables to Drive
# ============================================================
import os, shutil, json
import pandas as pd
from IPython.display import display

# ---- 1. Aggregate eval_results.json from every per-config run --------
eval_df = aggregate_eval_results(MULTI_CFG_OUT, split='test', dataloader_idx=0)
if eval_df.empty:
    print('No eval_results.json files found under', MULTI_CFG_OUT)
else:
    print(f'Aggregated {len(eval_df)} config(s) × {eval_df.shape[1]} metric(s).')
    display(eval_df)

summary_dir = os.path.join(DRIVE_SAVEPATH, 'summary')
os.makedirs(summary_dir, exist_ok=True)

if not eval_df.empty:
    # CSV for easy downstream ingestion + JSON for programmatic use
    eval_df.to_csv(os.path.join(summary_dir, 'aggregated_eval_results.csv'))
    with open(os.path.join(summary_dir, 'aggregated_eval_results.json'), 'w') as f:
        json.dump({k: v.dropna().to_dict() for k, v in eval_df.iterrows()}, f, indent=2, default=float)
    print(f'  → {summary_dir}/aggregated_eval_results.csv')

# ---- 2. Combined loss-curve plot across all configs ------------------
combined_path = os.path.join(summary_dir, 'combined_loss_curves.png')
res = plot_combined_loss_curves(
    MULTI_CFG_OUT, combined_path,
    metrics=('train_cnn_loss', 'val_cnn_loss'),
    title='Combined Train/Val CNN Loss — All Configurations',
)
if res is not None:
    print(f'  → {res}')
else:
    print('  Combined loss curves: no data to plot.')

# ---- 3. Mirror per-config figures + CSVs to Drive --------------------
drive_runs = os.path.join(DRIVE_SAVEPATH, 'multi_config_runs')
os.makedirs(drive_runs, exist_ok=True)
copied = 0
if os.path.exists(MULTI_CFG_OUT):
    for entry in sorted(os.listdir(MULTI_CFG_OUT)):
        src = os.path.join(MULTI_CFG_OUT, entry)
        if not os.path.isdir(src):
            continue
        dst = os.path.join(drive_runs, entry)
        if os.path.exists(dst):
            shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
        copied += 1
print(f'Mirrored {copied} per-config artifact dir(s) → {drive_runs}')

# ---- 4. Compact overview of what ended up on Drive -------------------
print('\nFinal outputs on Drive:')
for root, dirs, files in os.walk(DRIVE_SAVEPATH):
    depth = root[len(DRIVE_SAVEPATH):].count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or root}/')
    for fn in sorted(files)[:8]:
        print(f'{indent}  {fn}')
    if len(files) > 8:
        print(f'{indent}  … and {len(files) - 8} more')

# ---- 5. Quick-look best/worst configurations -------------------------
if not eval_df.empty:
    acc_cols = [c for c in eval_df.columns if 'acc' in c.lower() and 'retrieval' in c.lower()]
    if acc_cols:
        col = acc_cols[0]
        ordered = eval_df[col].dropna().sort_values(ascending=False)
        print(f'\nTop configs by {col}:')
        for name, v in ordered.items():
            print(f'  {name:<38s} {v:.4f}')